# Analyse de sensibilité — $H_{\text{local}}$

On fait varier chaque paramètre du modèle et on regarde :
1. Est-ce que le **classement** des sites change ? (corrélation de rang / Spearman)
2. Est-ce que le **gradient NW→SE** tient ?
3. Est-ce que **Marseille > Brest** tient ?

Si le classement est stable → le paramètre n'est pas critique (robustesse).
Si le classement bouge → le paramètre est sensible (à justifier ou calibrer).

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
import rasterio
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE = Path("../water_model/data/raw")
AQUEDUCT_BWS = BASE / "aqueduct" / "bws_raw.tif"
AQUEDUCT_GWS = BASE / "aqueduct" / "gws_raw.tif"
ERA5_FULL    = BASE / "era5" / "era5_monthly_1981_2022.nc"

SITES = {
    "Paris-Saclay": (48.73, 2.17), "Lyon": (45.76, 4.83),
    "Marseille": (43.30, 5.37), "Strasbourg": (48.57, 7.75),
    "Brest": (48.39, -4.49), "Rennes": (48.11, -1.68),
    "Bordeaux": (44.84, -0.58), "Montpellier": (43.61, 3.88),
}
NW = ['Brest', 'Rennes']
SE = ['Marseille', 'Montpellier', 'Lyon']

# --- Fonctions du modele (copiees de water_stress_H_local) ---

def extract_raster_point(raster_path, lat, lon):
    with rasterio.open(raster_path) as src:
        vals = list(src.sample([(lon, lat)]))
        val = float(vals[0][0])
        if src.nodata is not None and val == src.nodata:
            return np.nan
        return val

def compute_spei(precip_mm, pet_mm, dates, window=12,
                 baseline_start=1981, baseline_end=2010):
    D = precip_mm - pet_mm
    n = len(D)
    D_acc = np.full(n, np.nan)
    for i in range(window - 1, n):
        D_acc[i] = np.sum(D[i - window + 1 : i + 1])
    years, months = dates.year, dates.month
    mask_base = (years >= baseline_start) & (years <= baseline_end)
    monthly_params = {}
    for m in range(1, 13):
        mask_m = (months == m) & mask_base & ~np.isnan(D_acc)
        base_m = D_acc[mask_m]
        if len(base_m) < 10:
            monthly_params[m] = None; continue
        shift_m = -np.min(base_m) + 1.0
        try:
            c, _, scale = stats.fisk.fit(base_m + shift_m, floc=0)
            _, ks_pval = stats.kstest(base_m + shift_m, 'fisk', args=(c, 0, scale))
            monthly_params[m] = (c, scale, shift_m, ks_pval, len(base_m))
        except Exception:
            monthly_params[m] = None
    spei = np.full(n, np.nan)
    for i in range(n):
        if np.isnan(D_acc[i]): continue
        params = monthly_params.get(months[i])
        if params is None: continue
        c, scale, shift_m, _, _ = params
        prob = stats.fisk.cdf(D_acc[i] + shift_m, c, loc=0, scale=scale)
        prob = np.clip(prob, 1e-6, 1 - 1e-6)
        spei[i] = stats.norm.ppf(prob)
    ks_pvals = [p[3] for p in monthly_params.values() if p is not None]
    return {'spei': spei, 'ks_pvalue': np.mean(ks_pvals) if ks_pvals else 0.0}

def find_land_gridpoint(lat, lon, ds, threshold=0.60):
    td = 'valid_time' if 'valid_time' in ds.dims else 'time'
    lats, lons = ds.latitude.values, ds.longitude.values
    i_lat = int(np.argmin(np.abs(lats - lat)))
    i_lon = int(np.argmin(np.abs(lons - lon)))
    times = pd.DatetimeIndex(ds[td].values)
    month_idx = next(k for k in range(len(times)-1, -1, -1) if times[k].month == 7)
    lat_sl = slice(max(0, i_lat-1), min(len(lats), i_lat+2))
    lon_sl = slice(max(0, i_lon-1), min(len(lons), i_lon+2))
    pev_3x3 = ds['pev'].isel({td: month_idx, 'latitude': lat_sl, 'longitude': lon_sl})
    pet_3x3 = np.maximum(-pev_3x3.values.astype(float), 0.0)
    pet_center = pet_3x3[min(1, i_lat), min(1, i_lon)]
    pet_max = np.max(pet_3x3)
    ratio = pet_center / pet_max if pet_max > 0 else 1.0
    if ratio >= threshold:
        return lats[i_lat], lons[i_lon], False
    local_lats, local_lons = lats[lat_sl], lons[lon_sl]
    max_pos = np.unravel_index(np.argmax(pet_3x3), pet_3x3.shape)
    return float(local_lats[max_pos[0]]), float(local_lons[max_pos[1]]), True

# --- Precharger toutes les donnees brutes une seule fois ---

ds = xr.open_dataset(ERA5_FULL)
td = 'valid_time' if 'valid_time' in ds.dims else 'time'

coastal_map = {}
for name, (lat, lon) in SITES.items():
    blat, blon, coastal = find_land_gridpoint(lat, lon, ds)
    coastal_map[name] = (blat, blon, coastal)

RAW_DATA = {}
for name, (lat, lon) in SITES.items():
    bws_raw = extract_raster_point(AQUEDUCT_BWS, lat, lon)
    gws_raw = extract_raster_point(AQUEDUCT_GWS, lat, lon)
    pet_lat, pet_lon, _ = coastal_map[name]
    tp = ds['tp'].sel(latitude=lat, longitude=lon, method='nearest')
    t = pd.DatetimeIndex(tp[td].values)
    dy = t.days_in_month.values.astype(float)
    p = np.maximum(tp.values.astype(float) * dy * 1000, 0)
    pv = ds['pev'].sel(latitude=pet_lat, longitude=pet_lon, method='nearest')
    pet = np.maximum(-pv.values.astype(float) * dy * 1000, 0)
    sr12 = compute_spei(p, pet, t, window=12)
    sr3  = compute_spei(p, pet, t, window=3)
    # AI baseline
    years = t.year
    mask = (years >= 1981) & (years <= 2010)
    p_ann = np.sum(p[mask]) / 30
    pet_ann = np.sum(pet[mask]) / 30
    ai = p_ann / pet_ann if pet_ann > 0 else 999.0
    RAW_DATA[name] = {
        'bws_raw': bws_raw, 'gws_raw': gws_raw, 'ai': ai,
        'spei12': sr12['spei'], 'spei3': sr3['spei'], 'times': t,
    }

ds.close()
print(f"Donnees prechargees pour {len(RAW_DATA)} sites")


def compute_H(params, year=2022, month=7):
    """Calcule H pour tous les sites avec un jeu de parametres (v5)."""
    beta_0 = params['beta']
    alpha = params['alpha']
    bws_max = params['bws_max']
    gws_max = params['gws_max']
    k_arid = params['k_arid']
    ai_mid = params.get('ai_mid', 0.65)
    use_max = params.get('use_max', True)
    tau = params.get('tau', 5.0)
    c_asym = params.get('c_asym', 2.0)  # parametre mapping asymptotique
    dynamic_beta = params.get('dynamic_beta', True)

    def f_spei(val):
        """Mapping asymptotique non saturant: f = -SPEI / (c + |SPEI|)"""
        if np.isnan(val): return np.nan
        if val >= 0: return 0.0
        return float(-val / (c_asym + abs(val)))

    results = {}
    for name, rd in RAW_DATA.items():
        b = float(np.clip(rd['bws_raw'] / bws_max, 0, 1)) if not np.isnan(rd['bws_raw']) else np.nan
        g = float(np.clip(rd['gws_raw'] / gws_max, 0, 1)) if not np.isnan(rd['gws_raw']) else np.nan
        ai = rd['ai']
        a = 1.0 / (1.0 + (ai / ai_mid) ** k_arid) if ai > 0 else 1.0

        comps = {'bws': b, 'aridity': a, 'gws': g}
        num = sum(alpha[k] * v for k, v in comps.items() if v is not None and not np.isnan(v))
        den = sum(alpha[k] for k, v in comps.items() if v is not None and not np.isnan(v))
        s_struct = num / den if den > 0 else np.nan

        idx = next((i for i, tt in enumerate(rd['times'])
                     if tt.year == year and tt.month == month), None)
        if idx is None:
            results[name] = np.nan; continue

        d12 = f_spei(rd['spei12'][idx])
        d3  = f_spei(rd['spei3'][idx])
        if np.isnan(d12) and np.isnan(d3):
            s_conj = np.nan
        elif np.isnan(d3):
            s_conj = d12
        elif np.isnan(d12):
            s_conj = d3
        elif use_max:
            # softmax lissee
            e12 = np.exp(tau * d12)
            e3  = np.exp(tau * d3)
            s_conj = (d12 * e12 + d3 * e3) / (e12 + e3)
        else:
            s_conj = 0.6 * d12 + 0.4 * d3

        if np.isnan(s_struct) and np.isnan(s_conj):
            H = np.nan
        elif np.isnan(s_struct):
            H = s_conj
        elif np.isnan(s_conj):
            H = s_struct
        else:
            # beta dynamique
            if dynamic_beta and not np.isnan(s_conj):
                beta_t = beta_0 * (1.0 - s_conj)
            else:
                beta_t = beta_0
            H = beta_t * s_struct + (1 - beta_t) * s_conj
        results[name] = float(np.clip(H, 0, 1)) if not np.isnan(H) else np.nan
    return results

# Parametres de reference
REF_PARAMS = {
    'beta': 0.70, 'alpha': {'bws': 0.50, 'aridity': 0.36, 'gws': 0.14},
    'bws_max': 0.60, 'gws_max': 0.15,
    'k_arid': 3.0, 'ai_mid': 0.65, 'use_max': True,
    'tau': 5.0, 'c_asym': 2.0, 'dynamic_beta': True,
}

H_ref = compute_H(REF_PARAMS)
print("H reference (2022-07) :")
for name, h in sorted(H_ref.items(), key=lambda x: -x[1]):
    print(f"  {name:14s} : {h:.3f}")

Donnees prechargees pour 8 sites
H reference (2022-07) :
  Montpellier    : 0.637
  Marseille      : 0.635
  Lyon           : 0.585
  Paris-Saclay   : 0.563
  Strasbourg     : 0.536
  Rennes         : 0.426
  Bordeaux       : 0.401
  Brest          : 0.362


---
## 1. Sensibilité à $\beta$ (poids struct / conj)

$\beta$ varie de 0.50 (50/50) à 0.90 (90% structurel).
Question : le classement tient-il ?

In [2]:
from scipy.stats import spearmanr

ref_series = pd.Series(H_ref)
ref_rank = ref_series.rank()

betas = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90]
print(f'{"beta":>6s}  {"H_moy":>6s}  {"Spearman":>8s}  {"NW":>6s}  {"SE":>6s}  {"Grad":>5s}  {"M>B":>4s}')
print('-' * 55)

for b in betas:
    p = {**REF_PARAMS, 'beta': b}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    tag = ' <-- ref' if b == 0.70 else ''
    print(f'{b:6.2f}  {s.mean():6.3f}  {rho:8.3f}  {nw:6.3f}  {se:6.3f}  {grad:>5s}  {mb:>4s}{tag}')

  beta   H_moy  Spearman      NW      SE   Grad   M>B
-------------------------------------------------------
  0.50   0.536     0.976   0.417   0.628     OK    OK
  0.55   0.532     1.000   0.411   0.625     OK    OK
  0.60   0.527     1.000   0.405   0.623     OK    OK
  0.65   0.523     1.000   0.399   0.621     OK    OK
  0.70   0.518     1.000   0.394   0.619     OK    OK <-- ref
  0.75   0.514     0.976   0.388   0.617     OK    OK
  0.80   0.509     0.976   0.382   0.615     OK    OK
  0.85   0.505     0.976   0.376   0.613     OK    OK
  0.90   0.500     0.976   0.371   0.611     OK    OK


---
## 2. Sensibilité aux $\alpha$ (poids intra-structurel)

On teste 4 jeux de poids alternatifs.

In [3]:
alpha_sets = {
    'BWS domine':   {'bws': 0.70, 'aridity': 0.20, 'gws': 0.10},
    'Aridity domine': {'bws': 0.25, 'aridity': 0.60, 'gws': 0.15},
    'Egal':         {'bws': 0.34, 'aridity': 0.33, 'gws': 0.33},
    'Reference':    {'bws': 0.50, 'aridity': 0.36, 'gws': 0.14},
    'Sans GWS':     {'bws': 0.58, 'aridity': 0.42, 'gws': 0.00},
}

print(f'{"Config":>17s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}  {"Marseille":>9s}  {"Brest":>6s}')
print('-' * 70)

for label, alpha in alpha_sets.items():
    p = {**REF_PARAMS, 'alpha': alpha}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    tag = ' <--' if label == 'Reference' else ''
    print(f'{label:>17s}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}  {H["Marseille"]:9.3f}  {H["Brest"]:6.3f}{tag}')

           Config   H_moy  Spearman   Grad   M>B  Marseille   Brest
----------------------------------------------------------------------
       BWS domine   0.513     0.952     OK    OK      0.634   0.381
   Aridity domine   0.521     0.976     OK    OK      0.634   0.333
             Egal   0.536     0.976     OK    OK      0.646   0.354
        Reference   0.518     1.000     OK    OK      0.635   0.362 <--
         Sans GWS   0.506     0.952     OK    OK      0.627   0.361


---
## 3. Sensibilité aux seuils de rescaling (BWS_MAX, GWS_MAX)

In [4]:
print('=== BWS_MAX ===')
print(f'{"BWS_MAX":>8s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}')
print('-' * 40)
for bmax in [0.30, 0.40, 0.50, 0.60, 0.80, 1.00, 2.50, 5.00]:
    p = {**REF_PARAMS, 'bws_max': bmax}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    tag = ' <--' if bmax == 0.60 else ''
    print(f'{bmax:8.2f}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}{tag}')

print()
print('=== GWS_MAX ===')
print(f'{"GWS_MAX":>8s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}')
print('-' * 40)
for gmax in [0.10, 0.15, 0.20, 0.30, 0.50, 1.00, 5.00]:
    p = {**REF_PARAMS, 'gws_max': gmax}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    tag = ' <--' if gmax == 0.15 else ''
    print(f'{gmax:8.2f}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}{tag}')

=== BWS_MAX ===
 BWS_MAX   H_moy  Spearman   Grad   M>B
----------------------------------------
    0.30   0.549     0.952     OK    OK
    0.40   0.535     0.976     OK    OK
    0.50   0.527     0.976     OK    OK
    0.60   0.518     1.000     OK    OK <--
    0.80   0.507     1.000     OK    OK
    1.00   0.501     1.000     OK    OK
    2.50   0.485     0.905     OK    OK
    5.00   0.480     0.833     OK    OK

=== GWS_MAX ===
 GWS_MAX   H_moy  Spearman   Grad   M>B
----------------------------------------
    0.10   0.521     1.000     OK    OK
    0.15   0.518     1.000     OK    OK <--
    0.20   0.517     1.000     OK    OK
    0.30   0.514     1.000     OK    OK
    0.50   0.509     1.000     OK    OK
    1.00   0.503     1.000     OK    OK
    5.00   0.498     0.976     OK    OK


---
## 4. Sensibilité à l'exposant d'aridité ($k$)

$A = \frac{1}{1+(AI/0.65)^k}$ — $k$ contrôle la pente de la transition.

In [5]:
print(f'{"k":>4s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}  {"A_marseille":>11s}  {"A_brest":>8s}')
print('-' * 60)
for k in [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 8.0]:
    p = {**REF_PARAMS, 'k_arid': k}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    # Aridity values
    ai_m, ai_b = RAW_DATA['Marseille']['ai'], RAW_DATA['Brest']['ai']
    a_m = 1.0 / (1.0 + (ai_m / 0.65) ** k)
    a_b = 1.0 / (1.0 + (ai_b / 0.65) ** k)
    tag = ' <--' if k == 3.0 else ''
    print(f'{k:4.1f}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}  {a_m:11.3f}  {a_b:8.3f}{tag}')

   k   H_moy  Spearman   Grad   M>B  A_marseille   A_brest
------------------------------------------------------------
 1.0   0.530     1.000     OK    OK        0.613     0.269
 1.5   0.526     1.000     OK    OK        0.665     0.182
 2.0   0.523     1.000     OK    OK        0.714     0.119
 2.5   0.520     1.000     OK    OK        0.759     0.076
 3.0   0.518     1.000     OK    OK        0.798     0.047 <--
 4.0   0.515     0.976     OK    OK        0.862     0.018
 5.0   0.513     0.976     OK    OK        0.908     0.007
 8.0   0.510     0.976     OK    OK        0.975     0.000


---
## 5. max vs moyenne pondérée dans $S_{\text{conj}}$

$\max(f(\text{SPEI-12}), f(\text{SPEI-3}))$ vs $0.6 \cdot f(\text{SPEI-12}) + 0.4 \cdot f(\text{SPEI-3})$

In [6]:
for year in [2015, 2018, 2022]:
    print(f'=== Juillet {year} ===')
    H_max = compute_H({**REF_PARAMS, 'use_max': True}, year=year)
    H_avg = compute_H({**REF_PARAMS, 'use_max': False}, year=year)
    s_max, s_avg = pd.Series(H_max), pd.Series(H_avg)
    rho, _ = spearmanr(s_max, s_avg)

    print(f'{"Site":>14s}  {"H_max":>6s}  {"H_avg":>6s}  {"delta":>6s}')
    for name in sorted(SITES.keys()):
        d = H_max[name] - H_avg[name]
        print(f'{name:>14s}  {H_max[name]:6.3f}  {H_avg[name]:6.3f}  {d:+6.3f}')

    nw_m = np.mean([H_max[x] for x in NW])
    se_m = np.mean([H_max[x] for x in SE])
    nw_a = np.mean([H_avg[x] for x in NW])
    se_a = np.mean([H_avg[x] for x in SE])
    print(f'  Spearman max vs avg : {rho:.3f}')
    print(f'  Gradient max : SE/NW = {se_m/nw_m:.2f}x  {"OK" if se_m > nw_m else "FAIL"}')
    print(f'  Gradient avg : SE/NW = {se_a/nw_a:.2f}x  {"OK" if se_a > nw_a else "FAIL"}')
    print()

=== Juillet 2015 ===
          Site   H_max   H_avg   delta
      Bordeaux   0.401   0.402  -0.001
         Brest   0.177   0.178  -0.001
          Lyon   0.436   0.216  +0.220
     Marseille   0.718   0.561  +0.157
   Montpellier   0.449   0.305  +0.144
  Paris-Saclay   0.573   0.376  +0.197
        Rennes   0.249   0.238  +0.011
    Strasbourg   0.544   0.365  +0.179
  Spearman max vs avg : 0.714
  Gradient max : SE/NW = 2.51x  OK
  Gradient avg : SE/NW = 1.73x  OK

=== Juillet 2018 ===
          Site   H_max   H_avg   delta
      Bordeaux   0.217   0.217  +0.000
         Brest   0.261   0.191  +0.069
          Lyon   0.175   0.162  +0.013
     Marseille   0.586   0.586  +0.000
   Montpellier   0.267   0.267  +0.000
  Paris-Saclay   0.436   0.258  +0.179
        Rennes   0.217   0.217  +0.000
    Strasbourg   0.543   0.367  +0.176
  Spearman max vs avg : 0.905
  Gradient max : SE/NW = 1.43x  OK
  Gradient avg : SE/NW = 1.66x  OK

=== Juillet 2022 ===
          Site   H_max   H_avg   

---
## 5b. Softmax lissée pour $S_{\text{conj}}$

$$S_{\text{conj}} = \frac{d_{12} \cdot e^{\tau d_{12}} + d_3 \cdot e^{\tau d_3}}{e^{\tau d_{12}} + e^{\tau d_3}}$$

- $\tau = 0$ → moyenne simple
- $\tau \to \infty$ → max pur
- On teste $\tau \in [0, 2, 5, 10, 20, 50]$

In [7]:
# ==================================================================
# 5b. Mapping asymptotique — test de c
# ==================================================================

print("=== Sensibilite au parametre c (mapping asymptotique) ===")
print(f'{"c":>5s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}  {"f(-3)":>6s}  {"f(-5)":>6s}')
print('-' * 55)
for c_val in [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]:
    p = {**REF_PARAMS, 'c_asym': c_val}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    f3 = 3.0 / (c_val + 3.0)
    f5 = 5.0 / (c_val + 5.0)
    tag = ' <--' if c_val == 2.0 else ''
    print(f'{c_val:5.1f}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}  {f3:6.3f}  {f5:6.3f}{tag}')

print()
print("=== beta dynamique vs fixe ===")
print(f'{"Mode":>16s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}')
print('-' * 50)
for mode, label in [(True, 'dynamique'), (False, 'fixe (0.70)')]:
    p = {**REF_PARAMS, 'dynamic_beta': mode}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    print(f'{label:>16s}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}')
    print(f'                  {" ".join(f"{name[:4]}={H[name]:.3f}" for name in sorted(SITES))}')

print()
print("=== softmax tau ===")
print(f'{"tau":>5s}  {"H_moy":>6s}  {"Spearman":>8s}  {"Grad":>5s}  {"M>B":>4s}')
print('-' * 40)
for tau_val in [0, 1, 2, 3, 5, 8, 10, 50]:
    p = {**REF_PARAMS, 'tau': float(tau_val)}
    H = compute_H(p)
    s = pd.Series(H)
    rho, _ = spearmanr(ref_series, s)
    nw = np.mean([H[x] for x in NW])
    se = np.mean([H[x] for x in SE])
    grad = 'OK' if se > nw else 'FAIL'
    mb = 'OK' if H['Marseille'] > H['Brest'] else 'FAIL'
    tag = ' <--' if tau_val == 5 else ''
    print(f'{tau_val:5.0f}  {s.mean():6.3f}  {rho:8.3f}  {grad:>5s}  {mb:>4s}{tag}')


=== Sensibilite au parametre c (mapping asymptotique) ===
    c   H_moy  Spearman   Grad   M>B   f(-3)   f(-5)
-------------------------------------------------------
  0.5   0.777     0.976     OK    OK   0.857   0.909
  1.0   0.656     0.976     OK    OK   0.750   0.833
  1.5   0.576     1.000     OK    OK   0.667   0.769
  2.0   0.518     1.000     OK    OK   0.600   0.714 <--
  2.5   0.475     0.976     OK    OK   0.545   0.667
  3.0   0.442     0.976     OK    OK   0.500   0.625
  4.0   0.394     0.952     OK    OK   0.429   0.556
  5.0   0.361     0.952     OK    OK   0.375   0.500

=== beta dynamique vs fixe ===
            Mode   H_moy  Spearman   Grad   M>B
--------------------------------------------------
       dynamique   0.518     1.000     OK    OK
                  Bord=0.401 Bres=0.362 Lyon=0.585 Mars=0.635 Mont=0.637 Pari=0.563 Renn=0.426 Stra=0.536
     fixe (0.70)   0.407     0.667     OK    OK
                  Bord=0.354 Bres=0.279 Lyon=0.303 Mars=0.748 Mont=0.478

---
## 5c. Stabilité du ranking extrême

Est-ce que les sites les plus contraints (top) et les moins contraints (bottom)
restent stables quand on perturbe les paramètres ?


In [8]:
# ==================================================================
# 5c. Stabilite du ranking aux extremes
# ==================================================================

from collections import Counter

# Definir les perturbations a tester
perturbations = [
    ('ref',          REF_PARAMS),
    ('beta=0.50',    {**REF_PARAMS, 'beta': 0.50}),
    ('beta=0.90',    {**REF_PARAMS, 'beta': 0.90}),
    ('BWS_MAX=0.40', {**REF_PARAMS, 'bws_max': 0.40}),
    ('BWS_MAX=1.00', {**REF_PARAMS, 'bws_max': 1.00}),
    ('k_arid=1.5',   {**REF_PARAMS, 'k_arid': 1.5}),
    ('k_arid=5.0',   {**REF_PARAMS, 'k_arid': 5.0}),
    ('c_asym=1.0',   {**REF_PARAMS, 'c_asym': 1.0}),
    ('c_asym=3.0',   {**REF_PARAMS, 'c_asym': 3.0}),
    ('tau=0',        {**REF_PARAMS, 'tau': 0.0}),
    ('tau=10',       {**REF_PARAMS, 'tau': 10.0}),
    ('BWS domine',   {**REF_PARAMS, 'alpha': {'bws': 0.70, 'aridity': 0.20, 'gws': 0.10}}),
    ('Arid domine',  {**REF_PARAMS, 'alpha': {'bws': 0.25, 'aridity': 0.60, 'gws': 0.15}}),
    ('beta fixe',    {**REF_PARAMS, 'dynamic_beta': False}),
]

for year in [2015, 2022]:
    print(f'=== Juillet {year} — stabilite du top/bottom ===')
    top1_counts = Counter()
    top3_counts = Counter()
    bot1_counts = Counter()
    bot3_counts = Counter()
    all_rankings = {}

    for label, params in perturbations:
        H = compute_H(params, year=year)
        ranked = sorted(H.items(), key=lambda x: -x[1])
        top1_counts[ranked[0][0]] += 1
        for name, _ in ranked[:3]:
            top3_counts[name] += 1
        bot1_counts[ranked[-1][0]] += 1
        for name, _ in ranked[-3:]:
            bot3_counts[name] += 1
        all_rankings[label] = [r[0] for r in ranked]

    n = len(perturbations)
    print(f'  Top 1 (plus contraint) :')
    for site, cnt in top1_counts.most_common():
        print(f'    {site:14s} : {cnt}/{n} ({cnt/n*100:.0f}%)')
    print(f'  Top 3 :')
    for site, cnt in top3_counts.most_common(5):
        print(f'    {site:14s} : {cnt}/{n} ({cnt/n*100:.0f}%)')
    print(f'  Bottom 1 (moins contraint) :')
    for site, cnt in bot1_counts.most_common():
        print(f'    {site:14s} : {cnt}/{n} ({cnt/n*100:.0f}%)')
    print(f'  Bottom 3 :')
    for site, cnt in bot3_counts.most_common(5):
        print(f'    {site:14s} : {cnt}/{n} ({cnt/n*100:.0f}%)')
    print()

    # Show full ranking table
    print(f'  Rankings complets :')
    print(f'{"Config":>16s}  {"1er":>14s}  {"2e":>14s}  {"3e":>14s}  ... {"dernier":>14s}')
    print('-' * 80)
    for label, ranking in all_rankings.items():
        print(f'{label:>16s}  {ranking[0]:>14s}  {ranking[1]:>14s}  {ranking[2]:>14s}  ... {ranking[-1]:>14s}')
    print()


=== Juillet 2015 — stabilite du top/bottom ===
  Top 1 (plus contraint) :
    Marseille      : 14/14 (100%)
  Top 3 :
    Marseille      : 14/14 (100%)
    Paris-Saclay   : 14/14 (100%)
    Strasbourg     : 12/14 (86%)
    Bordeaux       : 1/14 (7%)
    Montpellier    : 1/14 (7%)
  Bottom 1 (moins contraint) :
    Brest          : 14/14 (100%)
  Bottom 3 :
    Rennes         : 14/14 (100%)
    Brest          : 14/14 (100%)
    Bordeaux       : 12/14 (86%)
    Lyon           : 2/14 (14%)

  Rankings complets :
          Config             1er              2e              3e  ...        dernier
--------------------------------------------------------------------------------
             ref       Marseille    Paris-Saclay      Strasbourg  ...          Brest
       beta=0.50       Marseille    Paris-Saclay      Strasbourg  ...          Brest
       beta=0.90       Marseille    Paris-Saclay      Strasbourg  ...          Brest
    BWS_MAX=0.40       Marseille    Paris-Saclay      Strasbourg

---
## 5d. Effet du $\beta$ dynamique

Comparaison détaillée : distribution des scores, cas où le $\beta$ dynamique
change le classement ou une décision.


In [9]:
# ==================================================================
# 5d. Effet isole du beta dynamique
# ==================================================================

print('=== Distribution des scores : beta dynamique vs fixe ===')
print()

for year in [2015, 2018, 2022]:
    H_dyn = compute_H({**REF_PARAMS, 'dynamic_beta': True}, year=year)
    H_fix = compute_H({**REF_PARAMS, 'dynamic_beta': False}, year=year)

    # Compute beta_t for each site
    print(f'--- Juillet {year} ---')
    print(f'{"Site":>14s}  {"H_dyn":>6s}  {"H_fix":>6s}  {"delta":>7s}  {"beta_t":>6s}  {"Rang_dyn":>8s}  {"Rang_fix":>8s}  {"Swap":>5s}')
    print('-' * 80)

    # Rankings
    rank_dyn = sorted(H_dyn.items(), key=lambda x: -x[1])
    rank_fix = sorted(H_fix.items(), key=lambda x: -x[1])
    rk_dyn = {name: i+1 for i, (name, _) in enumerate(rank_dyn)}
    rk_fix = {name: i+1 for i, (name, _) in enumerate(rank_fix)}

    for name in sorted(SITES.keys()):
        hd = H_dyn[name]
        hf = H_fix[name]
        delta = hd - hf
        # Estimate beta_t: need S_conj
        # beta_t = 0.70 * (1 - s_conj)
        # H_fix = 0.70 * s_struct + 0.30 * s_conj
        # We can back-compute s_conj from H_fix and s_struct
        # But simpler: compute directly
        rd = RAW_DATA[name]
        idx = next((i for i, tt in enumerate(rd['times'])
                     if tt.year == year and tt.month == 7), None)
        if idx is None:
            continue
        spei12_val = rd['spei12'][idx]
        spei3_val = rd['spei3'][idx]
        c_a = 2.0
        def f_s(v):
            if np.isnan(v) or v >= 0: return 0.0
            return -v / (c_a + abs(v))
        d12 = f_s(spei12_val)
        d3 = f_s(spei3_val)
        if d12 > 0 or d3 > 0:
            tau = 5.0
            e12 = np.exp(tau * d12)
            e3 = np.exp(tau * d3)
            s_conj = (d12 * e12 + d3 * e3) / (e12 + e3)
        else:
            s_conj = 0.0
        beta_t = 0.70 * (1.0 - s_conj)
        swap = '*' if rk_dyn[name] != rk_fix[name] else ''
        print(f'{name:>14s}  {hd:6.3f}  {hf:6.3f}  {delta:+7.3f}  {beta_t:6.3f}  {rk_dyn[name]:>8d}  {rk_fix[name]:>8d}  {swap:>5s}')

    # Stats
    vals_dyn = list(H_dyn.values())
    vals_fix = list(H_fix.values())
    print(f'  Ecart moyen     : {np.mean([H_dyn[n]-H_fix[n] for n in SITES]):+.3f}')
    print(f'  Ecart max       : {max(abs(H_dyn[n]-H_fix[n]) for n in SITES):.3f}')
    print(f'  Ecart-type dyn  : {np.std(vals_dyn):.3f}')
    print(f'  Ecart-type fix  : {np.std(vals_fix):.3f}')
    n_swaps = sum(1 for n in SITES if rk_dyn[n] != rk_fix[n])
    print(f'  Swaps de rang   : {n_swaps}/{len(SITES)}')
    rho_df, _ = spearmanr(list(H_dyn.values()), list(H_fix.values()))
    print(f'  Spearman dyn/fix: {rho_df:.3f}')
    print()

# Decision impact : seuils
print('=== Impact sur les decisions (seuils) ===')
print()
SEUILS = {'Faible': (0, 0.33), 'Modere': (0.33, 0.55), 'Eleve': (0.55, 0.75), 'Critique': (0.75, 1.01)}

def classify(h):
    for label, (lo, hi) in SEUILS.items():
        if lo <= h < hi:
            return label
    return 'Critique'

for year in [2015, 2018, 2022]:
    H_dyn = compute_H({**REF_PARAMS, 'dynamic_beta': True}, year=year)
    H_fix = compute_H({**REF_PARAMS, 'dynamic_beta': False}, year=year)
    changes = []
    for name in sorted(SITES.keys()):
        cd = classify(H_dyn[name])
        cf = classify(H_fix[name])
        if cd != cf:
            changes.append((name, H_fix[name], cf, H_dyn[name], cd))
    if changes:
        print(f'  {year} — {len(changes)} site(s) changent de categorie :')
        for name, hf, cf, hd, cd in changes:
            print(f'    {name:14s} : {cf} (H={hf:.3f}) -> {cd} (H={hd:.3f})')
    else:
        print(f'  {year} — aucun changement de categorie')


=== Distribution des scores : beta dynamique vs fixe ===

--- Juillet 2015 ---
          Site   H_dyn   H_fix    delta  beta_t  Rang_dyn  Rang_fix   Swap
--------------------------------------------------------------------------------
      Bordeaux   0.401   0.354   +0.047   0.380         6         4      *
         Brest   0.177   0.183   -0.006   0.603         8         8       
          Lyon   0.436   0.263   +0.174   0.302         5         7      *
     Marseille   0.718   0.791   -0.074   0.221         1         1       
   Montpellier   0.449   0.413   +0.036   0.359         4         2      *
  Paris-Saclay   0.573   0.401   +0.172   0.237         2         3      *
        Rennes   0.249   0.266   -0.017   0.586         7         6      *
    Strasbourg   0.544   0.330   +0.213   0.241         3         5      *
  Ecart moyen     : +0.068
  Ecart max       : 0.213
  Ecart-type dyn  : 0.163
  Ecart-type fix  : 0.173
  Swaps de rang   : 6/8
  Spearman dyn/fix: 0.786

--- Juill

---
## 6. Synthèse

In [10]:
# Recapitulatif automatique
print('=== SYNTHESE SENSIBILITE (v5) ===')
print()

tests = [
    ('beta 0.50-0.90', [(0.50,), (0.60,), (0.70,), (0.80,), (0.90,)],
     lambda v: {**REF_PARAMS, 'beta': v[0]}),
    ('BWS_MAX 0.30-5.0', [(0.30,), (0.60,), (1.00,), (5.00,)],
     lambda v: {**REF_PARAMS, 'bws_max': v[0]}),
    ('GWS_MAX 0.10-5.0', [(0.10,), (0.15,), (0.50,), (5.00,)],
     lambda v: {**REF_PARAMS, 'gws_max': v[0]}),
    ('k_arid 1-8', [(1.0,), (2.0,), (3.0,), (5.0,), (8.0,)],
     lambda v: {**REF_PARAMS, 'k_arid': v[0]}),
    ('c_asym 0.5-5', [(0.5,), (1.0,), (2.0,), (3.0,), (5.0,)],
     lambda v: {**REF_PARAMS, 'c_asym': v[0]}),
    ('tau 0-50', [(0,), (1,), (5,), (10,), (50,)],
     lambda v: {**REF_PARAMS, 'tau': float(v[0])}),
    ('beta dyn vs fixe', [(True,), (False,)],
     lambda v: {**REF_PARAMS, 'dynamic_beta': v[0]}),
    ('max vs avg', [(True,), (False,)],
     lambda v: {**REF_PARAMS, 'use_max': v[0]}),
]

print(f'{"Parametre":>20s}  {"Spearman_min":>12s}  {"Spearman_max":>12s}  {"Sensible":>8s}')
print('-' * 60)
for label, vals, fn in tests:
    rhos = []
    for v in vals:
        p = fn(v)
        H = compute_H(p)
        s = pd.Series(H)
        rho, _ = spearmanr(ref_series, s)
        rhos.append(rho)
    rho_min, rho_max = min(rhos), max(rhos)
    sensible = 'OUI' if rho_min < 0.85 else 'non'
    print(f'{label:>20s}  {rho_min:12.3f}  {rho_max:12.3f}  {sensible:>8s}')


=== SYNTHESE SENSIBILITE (v5) ===

           Parametre  Spearman_min  Spearman_max  Sensible
------------------------------------------------------------
      beta 0.50-0.90         0.976         1.000       non
    BWS_MAX 0.30-5.0         0.833         1.000       OUI
    GWS_MAX 0.10-5.0         0.976         1.000       non
          k_arid 1-8         0.976         1.000       non
        c_asym 0.5-5         0.952         1.000       non
            tau 0-50         0.905         1.000       non
    beta dyn vs fixe         0.667         1.000       OUI
          max vs avg         0.976         1.000       non
